# ETL Fase 3: Procesamiento de Tablas de Detalle (Transaccionales)
Este segundo notebook tiene como objetivo realizar el proceso ETL de las tablas de detalle o transaccionales del proyecto (`Club_Standings`, `Players` y `Medalists`). A diferencia de los catálogos estáticos, estas tablas contienen los eventos, clasificaciones y vínculos dinámicos que dan vida al análisis de negocio. Es un requisito indispensable que las tablas maestras ya estén cargadas en la base de datos antes de ejecutar este flujo.

## 1. Importación de Librerías y Configuración del Entorno
Comenzamos preparando nuestro entorno de trabajo, importando las librerías estándar para el análisis de datos, la gestión de credenciales y la conexión al motor relacional.

In [ ]:
# Herramienta principal para la limpieza, cruce y transformación de grandes volúmenes de datos
import pandas as pd

# Herramientas del sistema para la gestión de rutas y lectura de archivos en diferentes sistemas operativos
import os
import sys

# Librerías esenciales para formatear cadenas de conexión y comunicarse directamente con SQL Server
import urllib.parse
from sqlalchemy import create_engine

# Módulos de seguridad para evitar exponer contraseñas, leyendo credenciales desde un archivo oculto .env
from dotenv import load_dotenv, find_dotenv

# Agregamos la ruta raíz del proyecto al sistema. 
# Esto asegura que Python pueda encontrar la carpeta de datos ('data/raw') y el archivo de entorno ('.env')
# sin importar desde qué subcarpeta estemos ejecutando este notebook.
sys.path.append(os.path.abspath('..'))

# Mensaje de validación para confirmar que la primera celda se ejecutó con éxito
print("Librerías importadas correctamente. El entorno está listo para procesar las tablas de detalle.")

Librerías importadas correctamente para las tablas de detalle.


## 2. Configuración y Conexión Segura a la Base de Datos
Al igual que en la fase anterior, necesitamos establecer un puente de comunicación entre este notebook y nuestro servidor de SQL Server. Dado que cada archivo `.ipynb` funciona en un entorno de memoria independiente, volvemos a cargar de forma segura nuestras credenciales desde el archivo oculto `.env`. Esto garantiza que los datos transaccionales que procesemos a continuación se inyecten exactamente en la misma base de datos donde previamente creamos nuestras tablas maestras.

In [ ]:
# 1. Buscamos y cargamos el archivo oculto .env en la memoria de este notebook
load_dotenv(find_dotenv())

# 2. Extraemos las credenciales sin exponerlas en el código fuente
server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')
driver = os.getenv('DB_DRIVER')

# 3. Control de seguridad: Validamos que el archivo .env se haya leído correctamente.
# Si falta información clave, interrumpimos el proceso inmediatamente para evitar conexiones fallidas.
if not server or not database:
    raise ValueError("Error Crítico: No se encontraron las credenciales en el archivo .env. Verifique la ruta.")

# 4. Construimos la cadena de conexión utilizando la autenticación nativa de Windows ('Trusted_Connection=yes')
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};Trusted_Connection=yes;"
)

# 5. Creamos el motor de base de datos (engine) que ejecutará las inserciones por nosotros
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Mensaje de confirmación visual para el usuario
print("Motor de base de datos configurado y listo para insertar las tablas de detalle.")

Motor de base de datos configurado y listo para insertar detalles.


## 3. Extracción de Datos de Detalle (Extract)
En este paso cargamos los tres archivos que contienen la actividad detallada y transaccional de la competencia. A diferencia de las tablas maestras, estos archivos registran la granularidad del evento: el listado completo de los jugadores participantes, el desglose de los puntos acumulados por cada organización en el campeonato y el registro histórico de los atletas que ganaron medallas. Al igual que antes, los importamos con el sufijo `_raw` para indicar que entran a memoria en su estado original y "sucio", listos para ser transformados en las siguientes secciones.

In [ ]:
# 1. Definimos la ruta de la carpeta donde se encuentran guardados los archivos originales
raw_data_path = '../data/raw/'

# 2. Cargamos los archivos CSV de detalle en la memoria RAM como DataFrames de pandas
# Usamos 'os.path.join' para asegurar la compatibilidad de las rutas en cualquier sistema operativo.

# Carga el listado de todos los jugadores profesionales inscritos con sus datos demográficos
df_players_raw = pd.read_csv(os.path.join(raw_data_path, '05_EWC2025_Player_Roster.csv'))

# Carga la tabla de posiciones del campeonato con los puntos y el dinero ganado por club
df_standings_raw = pd.read_csv(os.path.join(raw_data_path, '03_EWC2025_Club_Championship_Standings.csv'))

# Carga el registro histórico de los jugadores que alcanzaron el podio (oro, plata o bronce)
df_medalists_raw = pd.read_csv(os.path.join(raw_data_path, '02_EWC2025_Medalists.csv'))

# 3. Mensaje de control para asegurar que los archivos se leyeron correctamente
print("Archivos de detalle extraídos a memoria con éxito. Listos para la fase de transformación.")

Archivos de detalle extraídos a memoria con éxito.


## 4. Transformación y Homologación de Datos: Tabla de Jugadores (Transform)
Este bloque aborda uno de los desafíos más críticos en la preparación de datos: garantizar la **Integridad Referencial**. Dado que la tabla de jugadores se conecta directamente con el catálogo de Torneos mediante una Llave Foránea, la base de datos rechazará cualquier inserción si el nombre del videojuego tiene errores tipográficos, diferencias en los signos de puntuación o divisiones por género que no existan en la tabla maestra. 

Para solucionar esto, aplicamos una estrategia de limpieza profunda en varios pasos:
1. Eliminamos espacios en blanco invisibles accidentales.
2. Construimos un "diccionario de traducción" para corregir y unificar los nombres de los juegos de forma masiva.
3. Removemos jugadores duplicados evaluando sus nombres en minúsculas para evitar falsos positivos (ej. "Blaz" vs "BlaZ").
4. Como filtro de seguridad final, consultamos en tiempo real la base de datos de SQL Server, traemos la lista oficial de juegos permitidos y cruzamos la información. Si un registro sigue sin coincidir, se aísla de la carga final para evitar que el programa colapse.

In [ ]:
# 1. Renombrar columnas originales del CSV
# Ajustamos los nombres de las columnas para que hagan un match exacto con el esquema de SQL Server.
df_players = df_players_raw.rename(columns={
    'Player': 'Player_Name', 
    'Team': 'Organization_Name',   
    'Game': 'Game_Title',
    'Country': 'Country',
    'Age': 'Age', 
    'Experience_Years': 'Experience_Years', 
    'Prize_Earned_USD': 'Prize_Earned_USD', 
    'Social_Media_Followers_K': 'Social_Media_Followers_K'
})

# 2. LIMPIEZA DE ESPACIOS INVISIBLES
# El método '.str.strip()' elimina cualquier espacio en blanco al principio o al final de las cadenas de texto.
# Esto previene errores donde "JuegoA" y "JuegoA " sean detectados como títulos diferentes.
df_players['Player_Name'] = df_players['Player_Name'].str.strip()
df_players['Game_Title'] = df_players['Game_Title'].str.strip()

# 3. HOMOLOGACIÓN DE DATOS (Diccionario de Traducción)
# Creamos un diccionario donde la clave (izquierda) es cómo viene escrito en el archivo crudo de Kaggle,
# y el valor (derecha) es cómo lo guardamos oficialmente en nuestra tabla maestra de SQL Server.
mapeo_juegos = {
    'Call of Duty Black Ops 6': 'Call of Duty: Black Ops 6',
    'Fatal Fury City of the Wolves': 'Fatal Fury: City of the Wolves',
    'Call of Duty Warzone': 'Call of Duty: Warzone',
    'Mobile Legends Bang Bang Men': 'Mobile Legends: Bang Bang', # Unificamos las categorías por género
    'Mobile Legends Bang Bang Women': 'Mobile Legends: Bang Bang',
    'PUBG Battlegrounds': 'PUBG: Battlegrounds'
}

# '.replace()' aplica nuestro diccionario para traducir toda la columna en una sola operación masiva.
df_players['Game_Title'] = df_players['Game_Title'].replace(mapeo_juegos)

# 4. MANEJO DE DUPLICADOS AVANZADO
# Convertimos temporalmente todos los nombres a minúsculas en una nueva columna. 
# Esto nos permite detectar que "Shadow" y "shadow" son la misma persona y eliminar la fila repetida.
df_players['temp_lower'] = df_players['Player_Name'].str.lower()
df_players = df_players.drop_duplicates(subset=['temp_lower'], keep='first')
df_players = df_players.drop(columns=['temp_lower']) # Borramos la columna temporal para limpiar el DataFrame

# 5. VALIDACIÓN DE LLAVE FORÁNEA CONTRA SQL SERVER
# Nos conectamos a la base de datos y extraemos la lista oficial de videojuegos que ya existen en el catálogo.
juegos_bd_df = pd.read_sql("SELECT Game_Title FROM Tournaments", con=engine)
juegos_validos = juegos_bd_df['Game_Title'].tolist()

# Identificamos si, después de nuestra limpieza, sigue existiendo algún juego extraño en el archivo de jugadores.
juegos_huerfanos = df_players[~df_players['Game_Title'].isin(juegos_validos)]['Game_Title'].unique()

if len(juegos_huerfanos) > 0:
    print(f"⚠️ ADVERTENCIA: Aún se excluirán jugadores porque estos juegos no existen en la base de datos: {juegos_huerfanos}")
else:
    print("✅ ¡Perfecto! Todos los juegos del Roster coinciden con la Base de Datos.")

# Aplicamos un filtro de seguridad: solo conservamos a los jugadores cuyo juego sí exista en SQL Server.
df_players = df_players[df_players['Game_Title'].isin(juegos_validos)]

# 6. Selección de columnas finales para la base de datos
# Aseguramos que el orden estructural sea idéntico al de la tabla de destino.
columnas_players = ['Player_Name', 'Organization_Name', 'Game_Title', 'Country', 'Age', 'Experience_Years', 'Prize_Earned_USD', 'Social_Media_Followers_K']
df_players_final = df_players[columnas_players]

# Mensaje de control de calidad y visualización
print(f"Total de Jugadores integrados con éxito y listos para la inserción: {len(df_players_final)}")
df_players_final.head(5)

✅ ¡Perfecto! Todos los juegos del Roster coinciden con la Base de Datos.
Total de Jugadores integrados con éxito: 270


,Player_Name,Organization_Name,Game_Title,Country,Age,Experience_Years,Prize_Earned_USD,Social_Media_Followers_K
0,Kasssa,VK Gaming,Apex Legends,Brazil,24,5,0,45
1,QQ,VK Gaming,Apex Legends,China,22,4,0,120
2,LqDuD,VK Gaming,Apex Legends,China,21,3,0,80
3,Vaxlon,ROC Esports,Apex Legends,Philippines,25,6,0,35
4,Deeds,ROC Esports,Apex Legends,Australia,23,4,0,28


## 5. Transformación de la Tabla de Posiciones: Club Standings (Transform)
En este bloque preparamos la tabla que registra el rendimiento general de los clubes en el campeonato (puntos acumulados, victorias y premios financieros). A diferencia de la tabla de jugadores, este proceso es mucho más ligero y directo. Como en el notebook anterior ya nos encargamos de extraer y consolidar todos los nombres posibles de las organizaciones en nuestro catálogo maestro, tenemos la garantía de que cualquier `Organization_Name` presente en este archivo ya existe en la base de datos. Por lo tanto, nuestra única tarea aquí es estandarizar los nombres de las columnas y descartar cualquier dato residual que pudiera traer el archivo original.

In [ ]:
# 1. Renombrar columnas originales del CSV
# Ajustamos los nombres para que coincidan exactamente con la estructura de la base de datos.
# Es importante destacar que 'Organization_Name' actuará como Llave Foránea (FK) apuntando a la tabla 'Clubs'.
df_standings = df_standings_raw.rename(columns={
    'Organization': 'Organization_Name', 
    'Rank': 'Rank_Position', 
    'Total_Points': 'Total_Points',
    'Prize_Money_USD': 'Prize_Money_USD',
    'Tournament_Wins': 'Tournament_Wins', 
    'Top_8_Finishes': 'Top_8_Finishes'
})

# 2. Filtrado de seguridad (Selección estricta de columnas)
# Creamos una lista con los nombres exactos y en el orden correcto que espera SQL Server.
# Al filtrar el DataFrame con esta lista, eliminamos automáticamente cualquier columna extra, 
# vacía o irrelevante ("basura") que pudiera estar escondida en el archivo CSV original.
columnas_standings = ['Organization_Name', 'Rank_Position', 'Total_Points', 'Prize_Money_USD', 'Tournament_Wins', 'Top_8_Finishes']
df_standings_final = df_standings[columnas_standings]

# 3. Control de Volumetría y Visualización
# Verificamos cuántos registros de rendimiento de clubes están listos para ser inyectados.
print(f"Total de Posiciones de Clubes listas para cargar: {len(df_standings_final)}")

# Desplegamos los primeros registros para validar visualmente la estructura final
df_standings_final.head(5)

Total de Posiciones de Clubes listas para cargar: 24


,Organization_Name,Rank_Position,Total_Points,Prize_Money_USD,Tournament_Wins,Top_8_Finishes
0,Team Falcons,1,3850,7000000,2,5
1,Team Liquid,2,2400,4000000,2,4
2,Team Vitality,3,1950,3000000,1,4
3,Twisted Minds,4,1800,2250000,2,3
4,Virtus.pro,5,1600,2250000,0,5


## 6. Transformación y Validación Doble: Tabla de Medallistas (Transform)
En este bloque procesamos el registro histórico de los atletas ganadores. Esta tabla transaccional es la más estricta del modelo, ya que posee **dos Llaves Foráneas**: una apunta al catálogo de Torneos (`Game_Title`) y la otra a la tabla de Jugadores (`Player_Name`). 

Para garantizar una carga sin errores, ejecutamos un proceso de limpieza y homologación idéntico al de los jugadores (eliminación de espacios y traducción de nombres de videojuegos). Sin embargo, la pieza central de este bloque es la **validación de integridad referencial preventiva**. Antes de intentar enviar los datos a SQL Server, cruzamos esta tabla contra nuestra lista limpia de jugadores y contra el catálogo de juegos permitidos. Si un medallista no existe en el roster oficial, o si el juego en el que ganó no está registrado, lo descartamos de forma proactiva para evitar un colapso en la base de datos.

In [ ]:
# 1. Renombrar columnas originales del CSV
# Ajustamos los nombres para asegurar el emparejamiento exacto con las columnas en la base de datos.
df_medalists = df_medalists_raw.rename(columns={
    'Player': 'Player_Name',   # FK hacia la tabla Players
    'Event': 'Game_Title',     # FK hacia la tabla Tournaments
    'Medal': 'Medal_Type',
    'Role': 'Player_Role'      
})

# 2. LIMPIEZA Y HOMOLOGACIÓN DE JUEGOS
# Eliminamos espacios en blanco invisibles al inicio y final de las cadenas de texto
df_medalists['Player_Name'] = df_medalists['Player_Name'].str.strip()
df_medalists['Game_Title'] = df_medalists['Game_Title'].str.strip()

# Reutilizamos nuestro diccionario de traducción (creado en la celda de Players)
# para corregir las discrepancias en los nombres y categorías de los videojuegos.
mapeo_juegos = {
    'Call of Duty Black Ops 6': 'Call of Duty: Black Ops 6',
    'Fatal Fury City of the Wolves': 'Fatal Fury: City of the Wolves',
    'Call of Duty Warzone': 'Call of Duty: Warzone',
    'Mobile Legends Bang Bang Men': 'Mobile Legends: Bang Bang',
    'Mobile Legends Bang Bang Women': 'Mobile Legends: Bang Bang',
    'PUBG Battlegrounds': 'PUBG: Battlegrounds'
}

# Aplicamos la traducción de forma masiva
df_medalists['Game_Title'] = df_medalists['Game_Title'].replace(mapeo_juegos)

# 3. VALIDACIÓN DE INTEGRIDAD REFERENCIAL DOBLE (Filtros preventivos en Pandas)
# A. Validar que el jugador ganador realmente exista en nuestro DataFrame final de Players.
# Si un jugador está en el archivo de medallas pero no en el Roster, causaría un error de Foreign Key.
jugadores_validos = df_players_final['Player_Name'].tolist()
df_medalists = df_medalists[df_medalists['Player_Name'].isin(jugadores_validos)]

# B. Validar que el juego exista en la Base de Datos.
# (La variable 'juegos_validos' se consultó previamente en la celda 8 y la reutilizamos aquí).
df_medalists = df_medalists[df_medalists['Game_Title'].isin(juegos_validos)]

# 4. Filtrar columnas finales
# Estandarizamos la estructura final descartando cualquier columna no requerida por el modelo.
columnas_medalists = ['Player_Name', 'Game_Title', 'Medal_Type', 'Player_Role']
df_medalists_final = df_medalists[columnas_medalists]

# Control de Volumetría
print(f"Total de Medallistas 100% validados y listos para cargar: {len(df_medalists_final)}")

# Muestra visual para confirmar la estructura
df_medalists_final.head(5)

Total de Medallistas 100% validados y listos para cargar: 224


,Player_Name,Game_Title,Medal_Type,Player_Role
0,Kasssa,Apex Legends,Gold,Player
1,QQ,Apex Legends,Gold,Player
2,LqDuD,Apex Legends,Gold,Player
3,Vaxlon,Apex Legends,Silver,Player
4,Deeds,Apex Legends,Silver,Player


## 7. Ingesta de Datos de Detalle en SQL Server (Load)
Llegamos a la fase culminante de la preparación de datos original: la carga física de la información transaccional en nuestra base de datos. Para ello, reutilizamos nuestra función de inserción segura (Try-Except) que envía los registros mediante el parámetro `append`, respetando el esquema estructural de SQL Server. 

El aspecto más crítico de este bloque es el **orden de ejecución**. Debido a las reglas de Integridad Referencial que configuramos en la base de datos, debemos respetar una estricta jerarquía de dependencias (Llaves Foráneas):
1. **Club_Standings**: Se inserta primero, ya que solo depende del catálogo maestro de Clubes (previamente cargado).
2. **Players**: Se inserta en segundo lugar. Depende de que existan los Clubes y los Torneos.
3. **Medalists**: Se debe insertar **obligatoriamente al final**, ya que depende de los Torneos y de la tabla de Jugadores que acabamos de cargar un paso atrás.

In [ ]:
# 1. Función estandarizada de carga segura
# Reutilizamos la misma lógica de inyección de datos que usamos para las tablas maestras.
def cargar_tabla_detalle(df, table_name, engine):
    print(f"Iniciando carga de tabla: {table_name}...")
    
    # El bloque Try-Except captura los errores de base de datos (ej. violación de llaves foráneas)
    # y los reporta en la consola sin detener bruscamente la ejecución del notebook.
    try:
        # if_exists='append': Inserta las nuevas filas sin alterar la arquitectura de la tabla
        # index=False: Evita enviar el índice autogenerado por pandas
        df.to_sql(name=table_name, con=engine, if_exists='append', index=False)
        print(f"  -> Éxito: {len(df)} registros insertados en dbo.{table_name}.\n")
        
    except Exception as e:
        print(f"  -> ERROR CRÍTICO al cargar {table_name}:")
        print(e)
        print("\n")

# 2. Ejecución secuencial respetando la Jerarquía de Dependencias (FKs)
# Si alteramos este orden, SQL Server rechazará los datos para proteger la consistencia del modelo.

# Nivel 1: Tabla de posiciones (Solo requiere la tabla maestra Clubs)
cargar_tabla_detalle(df_standings_final, 'Club_Standings', engine)

# Nivel 2: Tabla de talento (Requiere las tablas maestras Clubs y Tournaments)
cargar_tabla_detalle(df_players_final, 'Players', engine)

# Nivel 3: Tabla histórica (Requiere las tablas maestras y TAMBIÉN la tabla transaccional Players)
cargar_tabla_detalle(df_medalists_final, 'Medalists', engine)

# 3. Mensaje de cierre del flujo principal
print("¡ETL de Tablas de Detalle completado con éxito! La Fase 3 (Preparación de datos) original está terminada.")

Iniciando carga de tabla: Club_Standings...
  -> Éxito: 24 registros insertados en dbo.Club_Standings.

Iniciando carga de tabla: Players...
  -> Éxito: 270 registros insertados en dbo.Players.

Iniciando carga de tabla: Medalists...
  -> Éxito: 224 registros insertados en dbo.Medalists.

¡ETL de Tablas de Detalle completado con éxito! La Fase 3 (Preparación de datos) original está terminada.
